In [1]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

/home/thomas/git/bracis/slm_stability_cl/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
device = "cuda:0" if torch.cuda.is_available() else "cpu"
model_name = "google/gemma-3-1b-it"

model = AutoModelForCausalLM.from_pretrained(
    model_name, 
    torch_dtype="auto"
)
model.to(device)
model.eval()

tokenizer = AutoTokenizer.from_pretrained(
    model_name
)

Loading weights: 100%|██████████| 340/340 [00:00<00:00, 6163.16it/s]


In [3]:
print(model.get_memory_footprint()/1e6)

1999.773954


In [4]:
GENERATION_CONFIGS = {
    "thinking": {
        "do_sample": True,
        "temperature": 0.1,
        "top_p": 0.95,
        "top_k": 20,
        "min_p": 0.0,
        "max_new_tokens": 2048,
    },
    "non_thinking": {
        "do_sample": True,
        "temperature": 0.1,
        "top_p": 0.8,
        "top_k": 20,
        "min_p": 0.0,
        "max_new_tokens": 1024,
    },
}

In [5]:
def format_prompt(
    question: str,
    task_type: str = "general",
) -> str:
    question = question.strip()

    if task_type == "math":
        return (
            f"{question}\n\n"
            "Please reason step by step, and put your final answer within \\boxed{}."
        )

    if task_type == "multiple_choice":
        return (
            f"{question}\n\n"
            "Please show your choice in the answer field with only the choice letter, "
            'e.g., "answer": "C".'
        )

    if task_type == "general":
        return question

    raise ValueError(f"Unsupported task_type: {task_type}")

In [6]:
question = "Choose an answer for the following question and give your reasons.\n\nQuestion:\nWhich figure of speech is used in this text?\nLuke's room is as tidy as an overgrown garden.\n\nChoices:\nA. verbal irony\nB. pun\n\nAnswer:"
task_type = "multiple_choice"

prompt = format_prompt(
    question=question,
    task_type=task_type
)

messages = [
    # {
    #     "role": "system",
    #     "content": [{"type": "text", "text": "You are a helpful assistant."},]
    # },
    {"role": "user", "content": prompt},
]

thinking = False
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=thinking,
)

print(text)

<bos><start_of_turn>user
Choose an answer for the following question and give your reasons.

Question:
Which figure of speech is used in this text?
Luke's room is as tidy as an overgrown garden.

Choices:
A. verbal irony
B. pun

Answer:

Please show your choice in the answer field with only the choice letter, e.g., "answer": "C".<end_of_turn>
<start_of_turn>model



In [7]:
inputs = tokenizer(
    text,
    return_tensors="pt",
).to(device)

mode = "thinking" if thinking else "non_thinking"

with torch.no_grad():
    output_ids = model.generate(
        **inputs,
        **GENERATION_CONFIGS[mode],
        pad_token_id=tokenizer.eos_token_id,
    )

generated_ids = output_ids[0][inputs["input_ids"].shape[-1]:]

response = tokenizer.decode(
    generated_ids,
    skip_special_tokens=True,
)

In [8]:
print(response.strip())

answer: B
